<a href="https://colab.research.google.com/github/felondrum/llm_driven_development_otus/blob/develop/hw2_student_version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 ДЗ №2: Работа с данными для LLM

## 🎯 Цель задания
После выполнения задания вы сможете:
- Предобрабатывать русскоязычные текстовые данные для LLM
- Работать с готовыми моделями HuggingFace для анализа тональности и NER
- Создавать эффективные промпты для LLM API
- Сравнивать качество работы разных подходов к анализу текста
- Формировать датасеты в формате instruction-following для fine-tuning
- Сохранять данные в правильных форматах для обучения LLM

## 📝 Структура задания
- **Часть 1** (35% оценки): Предобработка данных и работа с готовыми моделями
- **Часть 2** (35% оценки): LLM API и prompt engineering
- **Часть 3** (20% оценки): Подготовка данных для fine-tuning LLM
- **Часть 4** (10% оценки): Сравнительный анализ и визуализация

## ⚡ Критерии оценки
- Качество предобработки данных: 25%
- Корректность работы с готовыми моделями: 20%
- Эффективность промптов для LLM: 25%
- Правильность подготовки данных для fine-tuning: 20%
- Качество сравнительного анализа: 10%


## 🔧 Установка зависимостей

Установим необходимые библиотеки для работы с данными, готовыми моделями и LLM API.


In [1]:
%pip install pandas numpy matplotlib seaborn
%pip install transformers torch
%pip install openai>=1.0.0  # Для работы с OpenAI API
%pip install datasets
%pip install pymorphy2



  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 59.9 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=62189cc58141cc1a027b1f2426ea1a7eeb9563089c6d17b5d352aa10b253f15c
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt


In [2]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from typing import List, Dict, Tuple
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification
import warnings
warnings.filterwarnings('ignore')

# Настройка отображения
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

print("Библиотеки загружены успешно!")



Библиотеки загружены успешно!


## 📊 Часть 1: Предобработка данных и готовые модели (35% оценки)

### Задание 1.1: Анализ "грязного" датасета

Проанализируем реалистичный датасет с типичными проблемами: опечатки, разные регистры, лишние пробелы, эмодзи.


In [13]:
# Создаем "грязный" датасет с типичными проблемами реальных данных
# Включаем сложные случаи для демонстрации преимуществ LLM
raw_reviews = [
    # Простые случаи
    "отличный iphone 14 PRO!!!  купил в магазине  apple на тверской 😊. Камера супер",
    "УЖАСНОЕ обслуживание в сбербанке на красной площади.. менеджер иван петров вобще не помог(",

    # Сарказм и ирония (сложно для классических моделей)
    "Спасибо огромное сотрудникам МТС за то что 3 часа держали меня в очереди! Просто восхитительно 👏",
    "Какой замечательный сервис в Пятерочке - касса сломалась прямо передо мной, а персонал даже не извинился",

    # Смешанные эмоции
    "iPhone 13 хороший телефон, но цена кусается. В целом доволен покупкой в re:Store",
    "Ресторан Белуга красивый и атмосфера приятная, но официант Максим был невнимателен",

    # Сложная структура предложений
    "Хотя Tesla Model Y и дорогая машина, и сервис в Рольф Премиум иногда подводит, но в целом я очень доволен покупкой",
    "Не могу сказать что отель Ритц-Карлтон плохой, просто ожидал большего за такие деньги",

    # Контекстно-зависимые случаи
    "Заказал доставку в Яндекс.Еде из ресторана Дача на Рублевке - привезли холодное, но курьер Андрей был вежливый",
    "MacBook Pro 16 работает как часы уже год, покупал в iStore на Арбате у консультанта Елены",

    # Неоднозначные случаи
    "Сходил в кинотеатр Октябрь посмотреть новый фильм Marvel - ну такое себе, но попкорн вкусный был",
    "Обслуживание в банке ВТБ на Тверской оставляет желать лучшего, хотя менеджер Ольга старалась помочь",

    # Сложные именованные сущности
    "Купил новый Samsung Galaxy S24 Ultra в DNS на Ленинском проспекте, консультант Дмитрий Иванович всё объяснил",
    "Ужинал в ресторане White Rabbit на Смоленской площади - шеф-повар Владимир Мухин превзошел ожидания",

    # Опечатки и сленг
    "норм телек LG купил в эльдорадо, продавец норм чел был, всё рассказал про функции"
]

# TODO: Создайте DataFrame и проанализируйте проблемы в данных
# Создайте DataFrame из списка raw_reviews
# Добавьте колонку с правильными метками тональности для каждого отзыва
# Проанализируйте и выведите список проблем, которые вы видите в данных
# Подумайте: какие проблемы могут повлиять на качество анализа?

# Ваш код здесь:

df = pd.DataFrame(raw_reviews, columns=['review'])

sentiment_labels = [
    'positive',  # 1
    'negative',  # 2
    'negative',  # 3
    'negative',  # 4
    'positive',  # 5
    'mixed',     # 6
    'positive',  # 7
    'negative',  # 8
    'mixed',     # 9
    'positive',  # 10
    'mixed',     # 11
    'mixed',     # 12
    'positive',  # 13
    'positive',  # 14
    'positive'   # 15
]

df['true_sentiment'] = sentiment_labels

df['review_length'] = df['review'].str.len()
df['word_count'] = df['review'].str.split().str.len()
df['char_count'] = df['review'].str.len()
df['unique_words'] = df['review'].apply(lambda x: len(set(x.split())))

# 4. Анализ проблем в данных
print("="*80)
print("АНАЛИЗ ПРОБЛЕМ В ДАННЫХ")
print("="*80)

print("\n1. СТАТИСТИКА ПО ДАННЫМ:")
print(f"   Всего отзывов: {len(df)}")
print(f"   Распределение тональности:")
print(df['true_sentiment'].value_counts())
print(f"\n   Средняя длина отзыва: {df['review_length'].mean():.0f} символов")
print(f"   Среднее количество слов: {df['word_count'].mean():.1f}")


# Проблемы:
print("\n1: Проблемы, которые могут повлиять на качество анализа")
print("\n1: Сарказм/ирония")
print("\n2: Смшенные эмоции")
print("\n3: Специализированные названия")
print("\n      Примеры:")
print("\n iPhone 14 PRO", "Samsung Galaxy S24 Ultra", "Tesla Model Y",
    "MacBook Pro 16", "re:Store", "DNS", "iStore", "Рольф Премиум")
print("\n4: Опечатки/сленг")
print("\n Примеры: вобще, норм, телек")
print("\n5: Эмодзи/нестандартная пунктуация(скобочки-эмодзи)")




АНАЛИЗ ПРОБЛЕМ В ДАННЫХ

1. СТАТИСТИКА ПО ДАННЫМ:
   Всего отзывов: 15
   Распределение тональности:
true_sentiment
positive    7
negative    4
mixed       4
Name: count, dtype: int64

   Средняя длина отзыва: 94 символов
   Среднее количество слов: 14.9

1: Проблемы, которые могут повлиять на качество анализа

1: Сарказм/ирония

2: Смшенные эмоции

3: Специализированные названия

      Примеры:

 iPhone 14 PRO Samsung Galaxy S24 Ultra Tesla Model Y MacBook Pro 16 re:Store DNS iStore Рольф Премиум

4: Опечатки/сленг

 Примеры: вобще, норм, телек

5: Эмодзи/нестандартная пунктуация


### Задание 1.2: Очистка и нормализация данных


In [18]:
def clean_text(text: str) -> str:
    """
    Очистка и нормализация русскоязычного текста
    """
    # TODO: Реализуйте базовую очистку текста
    # Подумайте над следующими аспектами:
    # - Как убрать эмодзи и специальные символы?
    # - Как нормализовать пробелы и отступы?
    # - Нужно ли исправлять регистр? Как?
    # - Что делать с повторяющимися знаками препинания?
    # - Как разделить слитно написанные слова (например, iPhone14)?

    # Используйте регулярные выражения (модуль re)
    # Ваш код здесь:

    # Базовая очистка русскоязычного текста
    if not text or pd.isna(text):
        return ""

    # Приводим к строковому типу
    text = str(text)

    # Удаляем эмодзи и специальные символы (оставляем буквы, цифры, пробелы и базовую пунктуацию)
    text = re.sub(r"[^\w\s\-.,!?;:()]", "", text)

    # Заменяем все виды пробелов на обычный пробел
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    # Убираем повторяющиеся знаки препинания
    text = re.sub(r"[.]{2,}", ".", text)
    text = re.sub(r"[!]{2,}", "!", text)
    text = re.sub(r"[?]{2,}", "?", text)

    # Разбиваем на предложения и корректируем регистр
    sentences = re.split(r'([.!?])', text)
    processed_sentences = []

    for i in range(0, len(sentences) - 1, 2):
        sentence = sentences[i].strip()
        punctuation = sentences[i + 1] if i + 1 < len(sentences) else ''

        if sentence:
            sentence = sentence[0].upper() + sentence[1:].lower() if len(sentence) > 1 else sentence.upper()
            processed_sentences.append(sentence + punctuation)

    # Добавляем последнее предложение без знака препинания
    if len(sentences) % 2 == 1 and sentences[-1].strip():
        last_sentence = sentences[-1].strip()
        last_sentence = last_sentence[0].upper() + last_sentence[1:].lower() if len(last_sentence) > 1 else last_sentence.upper()
        processed_sentences.append(last_sentence)

    text = ' '.join(processed_sentences)

    # Паттерн для разделения слов с цифрами
    text = re.sub(r'([a-zA-Zа-яА-Я]+)(\d+)', r'\1 \2', text)
    # Паттерн для разделения слов, написанных в CamelCase
    text = re.sub(r'([a-zа-я])([A-ZА-Я])', r'\1 \2', text)
    # Паттерн для разделения цифр и букв, идущих после цифр
    text = re.sub(r'(\d+)([a-zA-Zа-яА-Я]+)', r'\1 \2', text)

    # Исправление опечаток
    common_typos = {
    r'\bвобще\b': 'вообще',
    r'\bнорм\b': 'нормальный', #для конкретного сценария (текст 15)
    r'\bчел\b': 'человек',
    r'\bтелек\b': 'телефон',
    }

    for typo, correct in common_typos.items():
        text = re.sub(typo, correct, text, flags=re.IGNORECASE)

    # Удаляем лишние пробелы вокруг знаков препинания
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)
    text = re.sub(r'([.,!?;:])\s+', r'\1 ', text)

    # Удаляем одиночные скобки в конце предложения или с пробелами
    text = re.sub(r'\s*[)]+\s*', ' ', text)
    text = re.sub(r'\s*[(]+\s*', ' ', text)


    # Финальная нормализация пробелов
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text

# TODO: Примените функцию очистки к данным и сравните результаты
# Создайте новую колонку с очищенными текстами
# Сравните исходные и очищенные тексты

print("Тестирование функции очистки:")
print("До очистки:")
for one_index, one_text in enumerate(df["review"].head(15)):
    print(f"{one_index+1}. {one_text}")

# Применяем очистку к колонке с текстами
df["cleaned_text"] = df["review"].apply(clean_text)

print("\nПосле очистки:")
for one_index, one_text in enumerate(df["cleaned_text"].head(15)):
    print(f"{one_index+1}. {one_text}")

# Сравним длины текстов до и после очистки
print(f"\nСредняя длина до очистки: {df['review'].str.len().mean():.1f}")
print(f"Средняя длина после очистки: {df['cleaned_text'].str.len().mean():.1f}")

Тестирование функции очистки:
До очистки:
1. отличный iphone 14 PRO!!!  купил в магазине  apple на тверской 😊. Камера супер
2. УЖАСНОЕ обслуживание в сбербанке на красной площади.. менеджер иван петров вобще не помог(
3. Спасибо огромное сотрудникам МТС за то что 3 часа держали меня в очереди! Просто восхитительно 👏
4. Какой замечательный сервис в Пятерочке - касса сломалась прямо передо мной, а персонал даже не извинился
5. iPhone 13 хороший телефон, но цена кусается. В целом доволен покупкой в re:Store
6. Ресторан Белуга красивый и атмосфера приятная, но официант Максим был невнимателен
7. Хотя Tesla Model Y и дорогая машина, и сервис в Рольф Премиум иногда подводит, но в целом я очень доволен покупкой
8. Не могу сказать что отель Ритц-Карлтон плохой, просто ожидал большего за такие деньги
9. Заказал доставку в Яндекс.Еде из ресторана Дача на Рублевке - привезли холодное, но курьер Андрей был вежливый
10. MacBook Pro 16 работает как часы уже год, покупал в iStore на Арбате у консульт

### Задание 1.3: Использование готовых моделей HuggingFace


In [24]:
# TODO: Загрузите готовые модели HuggingFace для анализа тональности и NER
# Исследуйте HuggingFace Hub и найдите подходящие русскоязычные модели для:
# - Анализа тональности (sentiment analysis)
# - Извлечения именованных сущностей (NER)
#
# Используйте функцию pipeline() из библиотеки transformers
# Обратите внимание на параметры модели и токенизатора

# Ваш код для загрузки моделей:

print("Загрузка моделей HuggingFace...")
print("-" * 50)

# Модель для анализа тональности (sentiment analysis)
sentiment_pipeline = pipeline(
    model="seara/rubert-tiny2-russian-sentiment",
    task="text-classification",
    truncation=True,
    max_length=512
)

# Модель для извлечения именованных сущностей (NER)
ner_pipeline = pipeline(
            "ner",
            model="DeepPavlov/rubert-base-cased",
            aggregation_strategy="simple"
        )

print("Модели успешно загружены!")
print(f"Модель тональности: {sentiment_pipeline.model.config._name_or_path}")
print(f"Модель NER: {ner_pipeline.model.config._name_or_path}")
print("-" * 50)

def analyze_with_huggingface(texts: List[str]) -> List[Dict]:
    """
    Анализ текстов с помощью готовых моделей HuggingFace
    """
    # TODO: Реализуйте функцию анализа
    # Для каждого текста:
    # 1. Примените модель анализа тональности
    # 2. Примените модель NER
    # 3. Соберите результаты в структурированном виде
    # 4. Верните список словарей с результатами

    results = []

    for i, text in enumerate(texts):
        print(f"\n{'='*60}")
        print(f"ТЕКСТ {i+1}:")
        print(f"'{text}'")
        print(f"{'='*60}")

        # 1. Анализ тональности
        sentiment_result = sentiment_pipeline(text[:512])[0]

        print(f"\n📊 АНАЛИЗ ТОНАЛЬНОСТИ:")
        print(f"   Класс: {sentiment_result['label']}")
        print(f"   Уверенность: {sentiment_result['score']:.3f}")

        # 2. Извлечение именованных сущностей (NER)
        ner_results = ner_pipeline(text)

        print(f"\n🏷️  ИМЕНОВАННЫЕ СУЩНОСТИ (NER):")

        # Группируем сущности по типам
        entities_dict = {}

        if ner_results and len(ner_results) > 0:
            for entity in ner_results:
                entity_type = entity['entity_group']
                entity_word = entity['word']

                # Очистка от подтокенов
                if entity_word.startswith('##'):
                    entity_word = entity_word[2:]

                # Добавляем в словарь
                if entity_type not in entities_dict:
                    entities_dict[entity_type] = []
                if entity_word not in entities_dict[entity_type]:
                    entities_dict[entity_type].append(entity_word)

            # Выводим найденные сущности по типам
            type_mapping = {
                'PER': 'Люди (имена)',
                'PERSON': 'Люди (имена)',
                'LOC': 'Локации',
                'LOCATION': 'Локации',
                'ORG': 'Организации',
                'ORGANIZATION': 'Организации',
                'MISC': 'Разное',
                'PRODUCT': 'Продукты/бренды'
            }

            has_entities = False
            for entity_type, display_name in type_mapping.items():
                if entity_type in entities_dict:
                    print(f"   {display_name}: {', '.join(entities_dict[entity_type])}")
                    has_entities = True

            if not has_entities:
                print("   ❌ Сущности не найдены")

            # Выводим детальный список
            print(f"\n   Детальный список ({len(ner_results)} сущностей):")
            for entity in ner_results:
                print(f"   • {entity['word']} -> {entity['entity_group']} (score: {entity['score']:.3f})")

        else:
            print("   ❌ Сущности не найдены")

        # Формируем структурированный результат
        result = {
            'text': text,
            'sentiment': sentiment_result['label'],
            'sentiment_score': sentiment_result['score'],
            'entities': entities_dict,
            'ner_raw': ner_results
        }

        results.append(result)

    return results

# TODO: Протестируйте модели на очищенных данных
# Проанализируйте несколько текстов (для начала возьмите 3-5)
# Выведите результаты в понятном формате
# Проанализируйте качество работы моделей
print("=" * 60)
print("ТЕСТИРОВАНИЕ МОДЕЛЕЙ HUGGINGFACE")
print("=" * 60)

# Анализ текстов
results = analyze_with_huggingface(df['cleaned_text'])

print("=" * 60)
print("НАЧАЛО АНАЛИЗА ТЕКСТОВ")
print("=" * 60)

# Анализ текстов
results = analyze_with_huggingface(df['cleaned_text'])

# Финальный анализ качества
print("\n" + "=" * 60)
print("ИТОГОВЫЙ АНАЛИЗ КАЧЕСТВА")
print("=" * 60)

print("\n📈 СТАТИСТИКА РАСПОЗНАВАНИЯ СУЩНОСТЕЙ:")
print("-" * 50)

for i, result in enumerate(results):
    entities_count = len(result['entities'])
    total_entities = sum(len(v) for v in result['entities'].values())
    print(f"Текст {i+1}: Найдено {total_entities} сущностей ({len(result['entities'])} типов)")

    if result['entities']:
        for entity_type, entity_list in result['entities'].items():
            print(f"  - {entity_type}: {entity_list}")


Загрузка моделей HuggingFace...
--------------------------------------------------


Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: seara/rubert-tiny2-russian-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

...

BertForTokenClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                        

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Модели успешно загружены!
Модель тональности: seara/rubert-tiny2-russian-sentiment
Модель NER: DeepPavlov/rubert-base-cased
--------------------------------------------------
ТЕСТИРОВАНИЕ МОДЕЛЕЙ HUGGINGFACE

ТЕКСТ 1:
'Отличный iphone 14 pro! Купил в магазине apple на тверской. Камера супер'

📊 АНАЛИЗ ТОНАЛЬНОСТИ:
   Класс: positive
   Уверенность: 0.996

🏷️  ИМЕНОВАННЫЕ СУЩНОСТИ (NER):
   ❌ Сущности не найдены

   Детальный список (5 сущностей):
   • Отлич -> LABEL_1 (score: 0.563)
   • ##ный -> LABEL_0 (score: 0.549)
   • iphone 14 pro! -> LABEL_1 (score: 0.609)
   • Купил в -> LABEL_0 (score: 0.590)
   • магазине apple на тверской. Камера супер -> LABEL_1 (score: 0.608)

ТЕКСТ 2:
'Ужасное обслуживание в сбербанке на красной площади. Менеджер иван петров вообще не помог'

📊 АНАЛИЗ ТОНАЛЬНОСТИ:
   Класс: negative
   Уверенность: 0.913

🏷️  ИМЕНОВАННЫЕ СУЩНОСТИ (NER):
   ❌ Сущности не найдены

   Детальный список (5 сущностей):
   • Ужасное обслуживание в сб -> LABEL_1 (score: 0.583)
 

## 🤖 Часть 2: LLM API и Prompt Engineering (35% оценки)

### Задание 2.1: Создание эффективных промптов


In [ ]:
def create_prompts_for_llm() -> Dict[str, str]:
    """
    Создание базовых промптов для разных задач (один промпт на задачу)
    """
    # TODO: Создайте эффективные промпты для NER и sentiment analysis
    # Подумайте о структуре хорошего промпта:
    # - Четкое описание задачи
    # - Примеры входных и выходных данных
    # - Формат ответа (JSON, текст и т.д.)
    # - Особые требования (например, для русского языка)

    # Создайте промпты для:
    # 1. Извлечения именованных сущностей (NER)
    # 2. Анализа тональности (sentiment analysis)

    # Ваш код здесь:
    pass

# TODO: Протестируйте ваши промпты
# Выведите созданные промпты и оцените их качество

# TODO: Настройте OpenAI API
# Установите API ключ через переменные окружения
# Изучите документацию OpenAI API для Python



# TODO: Реализуйте функции для работы с OpenAI API
# Создайте функции для:
# 1. Вызова OpenAI API с промптом
# 2. Обработки ответа от API
# 3. Анализа текстов с помощью ваших промптов
#
# Подумайте о:
# - Обработке ошибок API
# - Формате запроса и ответа
# - Параметрах модели (temperature, max_tokens)
#
# Протестируйте на нескольких текстах из датасета



### Задание 2.2: Сравнение результатов HuggingFace vs LLM


In [ ]:
# TODO: Сравните результаты HuggingFace моделей с LLM на одних и тех же текстах
# Создайте сравнительный анализ:
# 1. Соберите результаты обеих подходов в структурированном виде
# 2. Сравните точность анализа тональности
# 3. Сравните качество извлечения сущностей
# 4. Проанализируйте время выполнения
# 5. Оцените простоту использования
#
# Создайте визуализации для сравнения:
# - Точность по разным метрикам
# - Время обработки
# - Количество найденных сущностей
#
# Сделайте выводы о том, когда лучше использовать каждый подход



## 📚 Часть 3: Подготовка данных для Fine-tuning LLM (20% оценки)

### Задание 3.1: Создание instruction-following датасета


In [ ]:
### Задание 2.3: Анализ сложных случаев

# Выберем специально сложные примеры для демонстрации преимуществ LLM
complex_cases = [
    "Спасибо огромное сотрудникам МТС за то что 3 часа держали меня в очереди! Просто восхитительно 👏",
    "iPhone 13 хороший телефон, но цена кусается. В целом доволен покупкой в re:Store",
    "Хотя Tesla Model Y и дорогая машина, и сервис в Рольф Премиум иногда подводит, но в целом я очень доволен покупкой",
    "норм телек LG купил в эльдорадо, продавец норм чел был, всё рассказал про функции"
]

print("Анализ сложных случаев:")
print("=" * 60)

# TODO: Сравните результаты HuggingFace и OpenAI на сложных случаях
# for i, text in enumerate(complex_cases):
#     print(f"\nПример {i+1}: {text}")
#     # hf_result = sentiment_pipeline(text)
#     # openai_result = analyze_with_openai([text])
#     # print(f"HuggingFace: {hf_result}")
#     # print(f"OpenAI: {openai_result}")





In [ ]:
### Задание 2.4: Количественное сравнение точности

# Создаем расширенный набор для тестирования с правильными ответами
test_cases_with_labels = [
    # Сарказм и ирония - должны быть NEGATIVE
    ("Спасибо огромное сотрудникам МТС за то что 3 часа держали меня в очереди! Просто восхитительно 👏", "NEGATIVE"),
    ("Какой замечательный сервис в Пятерочке - касса сломалась прямо передо мной, а персонал даже не извинился", "NEGATIVE"),

    # Смешанные эмоции - должны быть NEUTRAL или зависеть от преобладающего тона
    ("iPhone 13 хороший телефон, но цена кусается. В целом доволен покупкой в re:Store", "NEUTRAL"),
    ("Ресторан Белуга красивый и атмосфера приятная, но официант Максим был невнимателен", "NEUTRAL"),

    # Сложные структуры - требуют понимания контекста
    ("Хотя Tesla Model Y и дорогая машина, и сервис в Рольф Премиум иногда подводит, но в целом я очень доволен покупкой", "POSITIVE"),
    ("Не могу сказать что отель Ритц-Карлтон плохой, просто ожидал большего за такие деньги", "NEUTRAL"),

    # Неформальная речь и сленг
    ("норм телек LG купил в эльдорадо, продавец норм чел был, всё рассказал про функции", "POSITIVE"),
    ("Сходил в кинотеатр Октябрь посмотреть новый фильм Marvel - ну такое себе, но попкорн вкусный был", "NEUTRAL"),

    # Простые случаи для контроля
    ("отличный iphone 14 PRO!!! купил в магазине apple на тверской 😊. Камера супер", "POSITIVE"),
    ("УЖАСНОЕ обслуживание в сбербанке на красной площади.. менеджер иван петров вобще не помог(", "NEGATIVE")
]

# TODO: Рассчитайте точность для каждой модели
# hf_correct = 0
# openai_correct = 0
# total = len(test_cases_with_labels)



In [ ]:
### Задание 2.5: Визуализация сравнения моделей

import matplotlib.pyplot as plt
import numpy as np

# TODO: Создайте визуализацию сравнения точности моделей
# plt.figure(figsize=(12, 8))
# # Создайте графики сравнения

In [ ]:
def create_instruction_dataset(df: pd.DataFrame) -> List[Dict]:
    """
    Создание датасета в формате instruction-following для fine-tuning LLM
    """
    # TODO: Создайте структурированный датасет для fine-tuning LLM
    # Подумайте о структуре instruction-following датасета:
    # - Какие поля должны быть в каждом примере?
    # - Как сформулировать инструкции для модели?
    # - Какие типы задач включить (sentiment, NER, etc.)?
    # - Как структурировать ответы модели?
    #
    # Создайте несколько примеров для разных задач

    pass

# TODO: Протестируйте созданный датасет
# Создайте и проанализируйте instruction dataset
# Выведите примеры в читаемом формате
# Проанализируйте распределение типов задач



### Задание 3.2: Сериализация данных в формате для LLM платформ


In [ ]:
import json

# TODO: Реализуйте сохранение данных в форматах для fine-tuning
# Создайте функции для сохранения данных в форматах:
# 1. JSONL формат для OpenAI fine-tuning API
# 2. CSV формат для общего использования
#
# Изучите требования к форматам:
# - Какая структура нужна для OpenAI fine-tuning?
# - Как правильно структурировать messages?
# - Какие поля обязательны?
#
# Протестируйте сохранение и загрузку данных


